# Database Explorer & Visualizer

This notebook provides an interactive SQL interface and plotting utility to explore the VD datasets.
It connects to the datasets using `duckdb_connection()` from `scripts/lib/data_loaders.py`.

Refer to `docs/DATA_DESIGN.md` for schema details.

In [ ]:
import os, sys

# ── environment detection ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Google Drive mount (Colab only) ──────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    # Local (VSCode/Jupyter): walk up if notebook lives in <root>/notebooks/
    _here = os.path.abspath(os.getcwd())
    _DEFAULT_ROOT = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
%cd {PROJECT_ROOT}
print('PROJECT_ROOT =', PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.express as px

# Import DuckDB connection
from scripts.lib.data_loaders import duckdb_connection

## 1. Initialize Connection

In [ ]:
print("Initializing DuckDB connection...")
con = duckdb_connection()

# Show available views (datasets)
try:
    tables = con.execute("SHOW TABLES").df()
    print("Available tables/views:")
    display(tables)
except Exception as e:
    print(f"Error checking tables: {e}")

## 2. Interactive Explorer GUI

In [ ]:
class DBExplorer:
    def __init__(self, con):
        self.con = con
        self.current_df = None
        self._build_gui()
        
    def _build_gui(self):
        # SQL Text Area
        self.sql_input = widgets.Textarea(
            value="SELECT * FROM vd_dim LIMIT 10;",
            placeholder="Enter SQL query here...",
            description="SQL:",
            layout=widgets.Layout(width="95%", height="100px")
        )
        
        # Execute Button
        self.exec_btn = widgets.Button(
            description="Execute Query",
            button_style="primary",
            icon="play",
            layout=widgets.Layout(margin="10px 0px 10px 100px")
        )
        self.exec_btn.on_click(self._on_execute)
        
        # Output Area
        self.out = widgets.Output()
        
        # Plotting Controls
        self.plot_type = widgets.Dropdown(
            options=["Table Only", "Line", "Bar", "Scatter", "Histogram"],
            value="Table Only",
            description="Plot Type:"
        )
        self.x_axis = widgets.Dropdown(description="X Axis:")
        self.y_axis = widgets.Dropdown(description="Y Axis:")
        self.color = widgets.Dropdown(description="Color:")
        
        self.plot_btn = widgets.Button(
            description="Generate Plot",
            button_style="info",
            icon="chart-bar"
        )
        self.plot_btn.on_click(self._on_plot)
        
        # Plot Output Area
        self.plot_out = widgets.Output()
        
        # Layout
        query_box = widgets.VBox([self.sql_input, self.exec_btn])
        plot_controls = widgets.HBox([self.plot_type, self.x_axis, self.y_axis, self.color, self.plot_btn])
        
        self.ui = widgets.VBox([
            widgets.HTML(value="<h3>SQL Query Interface</h3>"),
            query_box,
            self.out,
            widgets.HTML(value="<hr><h3>Visualization Options</h3>"),
            plot_controls,
            self.plot_out
        ])
        
    def _on_execute(self, b):
        with self.out:
            clear_output()
            query = self.sql_input.value.strip()
            if not query:
                print("Please enter a SQL query.")
                return
            
            try:
                print(f"Executing...\n")
                self.current_df = self.con.execute(query).df()
                
                # Show results summary
                print(f"Total rows returned: {len(self.current_df)}")
                display(self.current_df.head(100))
                if len(self.current_df) > 100:
                    print(f"... and {len(self.current_df) - 100} more rows.")
                
                # Update dropdowns for plotting
                cols = ["None"] + list(self.current_df.columns)
                self.x_axis.options = cols
                self.y_axis.options = cols
                self.color.options = cols
                
                if len(cols) > 1:
                    self.x_axis.value = cols[1]
                if len(cols) > 2:
                    self.y_axis.value = cols[2]
                
            except Exception as e:
                print(f"Error: {e}")
                
    def _on_plot(self, b):
        with self.plot_out:
            clear_output()
            if self.current_df is None or self.current_df.empty:
                print("No data available. Execute a query first.")
                return
                
            ptype = self.plot_type.value
            x_col = self.x_axis.value if self.x_axis.value != "None" else None
            y_col = self.y_axis.value if self.y_axis.value != "None" else None
            color_col = self.color.value if self.color.value != "None" else None
            
            if ptype == "Table Only":
                print("Select a plot type to visualize.")
                return
                
            try:
                if ptype == "Line":
                    fig = px.line(self.current_df, x=x_col, y=y_col, color=color_col)
                elif ptype == "Bar":
                    fig = px.bar(self.current_df, x=x_col, y=y_col, color=color_col)
                elif ptype == "Scatter":
                    fig = px.scatter(self.current_df, x=x_col, y=y_col, color=color_col)
                elif ptype == "Histogram":
                    fig = px.histogram(self.current_df, x=x_col, y=y_col, color=color_col)
                
                fig.update_layout(template="plotly_white")
                fig.show()
            except Exception as e:
                print(f"Plotting Error: {e}")
                
    def display(self):
        display(self.ui)

# Instantiate and display the explorer
explorer = DBExplorer(con)
explorer.display()

## 3. Example Queries
Refer to `docs/DATA_DESIGN.md` for complete schema definitions.

### View VD Dimension Table
```sql
SELECT * FROM vd_dim 
WHERE freeway = 'N1' AND direction = 'S' 
ORDER BY mileage_m 
LIMIT 10;
```

### Aggregate 1-min Station Data (Daily Metrics)
```sql
SELECT 
    vdid,
    COUNT(*) as obs_count,
    AVG(speed_x10)/10.0 as mean_speed,
    SUM(vol_s + vol_l + vol_t) as total_vol
FROM station_1m
WHERE source_date = '2024-04-01' AND freeway = 'N1'
GROUP BY vdid
ORDER BY total_vol DESC
LIMIT 10;
```

### Accident Counts by Freeway
```sql
SELECT 
    freeway,
    direction,
    COUNT(*) as accident_count
FROM accidents_raw
GROUP BY freeway, direction
ORDER BY accident_count DESC;
```